In [ ]:
import pandas as pd

from utils import data_utils as du
from utils import plot_utils as pu
from utils import stats_utils as su

cities = du.load_major_us_cities()
sf_raw = cities["sf_Existing_Buildings_Energy_Performance_Ordinance_Report_20260114"]
sea_raw = cities[
    "Seattle_Benchmarking_Performance_Ranges_by_Building_Type_2015-Present_20260125"
]
bos_raw = cities["boston_energy_raw"]

end_year = 2023
star_year_I = 2017
star_year_II = 2018
MAX_SITE_EUI = 1000

energy_data = du.load_data()
concurrent_df = du.concurrent_buildings(
    input_df=energy_data,
    start_year=2016,
    end_year=end_year,
)

2026-03-01 07:15:53,689 [INFO] Loaded Seattle_Benchmarking_Performance_Ranges_by_Building_Type_2015-Present_20260125.csv → (34699, 46)
2026-03-01 07:15:53,778 [INFO] Loaded sf_Existing_Buildings_Energy_Performance_Ordinance_Report_20260114.csv → (28243, 34)
2026-03-01 07:16:23,457 [INFO] Loaded Boston_data folder → (35496, 96)


In [6]:
sf_chi = du.sf_to_chicago(sf_raw)
sea_chi = du.seattle_to_chicago(sea_raw)
sf_chi = sf_chi.assign(City="San Francisco")
sea_chi = sea_chi.assign(City="Seattle")
bos = du.harmonize_boston_columns(bos_raw)
bos_chi = du.boston_to_chicago(bos).assign(City="Boston")
bos_chi["Data Year"] = pd.to_numeric(bos_chi["Data Year"]) - 1

## DiD

In [ ]:
did_df = su.build_multi_city_did_df(
    chicago_df=concurrent_df,
    other_city_dfs={
        "San Francisco": sf_chi,
        "Seattle": sea_chi,
        "Boston": bos_chi,
    },
    start_year=2016,
    end_year=end_year,
    post_start_year=2020,
    outcome_col="Site EUI (kBtu/sq ft)",
)

/project/src/utils/stats_utils.py:420: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  did_df = pd.concat(frames, ignore_index=True)
/project/src/utils/stats_utils.py:420: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  did_df = pd.concat(frames, ignore_index=True)
/project/.venv/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


### three cities as baseline on ALL building types

In [8]:
did_df = did_df.copy()

did_df["Site EUI (kBtu/sq ft)"] = pd.to_numeric(
    did_df["Site EUI (kBtu/sq ft)"], errors="coerce"
).astype("float64")

for c in ["Post", "LowRating", "Interaction"]:
    did_df[c] = pd.to_numeric(did_df[c], errors="coerce").fillna(0).astype("int64")

did_df["Data Year"] = pd.to_numeric(did_df["Data Year"], errors="coerce")
did_df = did_df.dropna(subset=["Data Year"])
did_df["Data Year"] = did_df["Data Year"].astype("int64")

did_df["Primary Property Type"] = (
    did_df["Primary Property Type"].astype(str).str.strip().str.lower()
)
if "City" in did_df.columns:
    did_df["City"] = did_df["City"].astype("object")

did_df = did_df.dropna(
    subset=["Site EUI (kBtu/sq ft)", "ln_FloorArea", "Primary Property Type"]
)

#### filter by concurrent

In [9]:
sf_concurrent = du.concurrent_buildings(
    input_df=sf_chi,
    start_year=2016,
    end_year=2024,
    id_col="Address",
    status_col="__IGNORE_STATUS__",
)

sea_concurrent = du.concurrent_buildings(
    input_df=sea_chi,
    start_year=2016,
    end_year=2024,
    id_col="Address",
    status_col="__IGNORE_STATUS__",
)

bos_concurrent = du.concurrent_buildings(
    input_df=bos_chi,
    start_year=2016,
    end_year=2024,
    id_col="Address",
    status_col="__IGNORE_STATUS__",
)

In [71]:
did_df_lvl = did_df.copy()

did_df_lvl["Site EUI (kBtu/sq ft)"] = pd.to_numeric(
    did_df_lvl["Site EUI (kBtu/sq ft)"], errors="coerce"
).astype("float64")

for c in ["Post", "LowRating", "Interaction"]:
    did_df_lvl[c] = (
        pd.to_numeric(did_df_lvl[c], errors="coerce").fillna(0).astype("int64")
    )

did_df_lvl["ln_FloorArea"] = pd.to_numeric(
    did_df_lvl["ln_FloorArea"], errors="coerce"
).astype("float64")

did_df_lvl["Data Year"] = pd.to_numeric(did_df_lvl["Data Year"], errors="coerce")
did_df_lvl = did_df_lvl.dropna(subset=["Data Year"])
did_df_lvl["Data Year"] = did_df_lvl["Data Year"].astype("int64")

did_df_lvl["Primary Property Type"] = (
    did_df_lvl["Primary Property Type"].astype(str).str.strip().str.lower()
)
if "City" in did_df_lvl.columns:
    did_df_lvl["City"] = did_df_lvl["City"].astype("object")

did_df_lvl = did_df_lvl.dropna(
    subset=[
        "Site EUI (kBtu/sq ft)",
        "ln_FloorArea",
        "Primary Property Type",
        "Data Year",
    ]
)

In [11]:
model_eui_lvl = su.run_did_regression(
    did_df_lvl,
    "Site EUI (kBtu/sq ft)",
    include_data_year=True,
)

su.summarize_did_results(
    model_eui_lvl,
    focus_terms=["Post", "LowRating", "Interaction"],
)

/project/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 101, but rank is 1
  warnings.warn('covariance of constraints does not have full '
2026-03-01 06:37:47,713 [INFO] Showing 9 selected coefficients (policy + energy types).


,coef,std_err,p_value,Significance
C(Q('Primary Property Type'))[T.data center],4.516089e+02,5.114310e+01,0.0000,***
C(Q('Primary Property Type'))[T.fitness center/health club/gym],2.027990e+01,1.253840e+01,0.1058,
C(Q('Primary Property Type'))[T.hospital (general medical & surgical)],2.200687e+02,6.146150e+01,0.0003,***
C(Q('Primary Property Type'))[T.laboratory],2.125650e+02,2.849640e+01,0.0000,***
C(Q('Primary Property Type'))[T.other - specialty hospital],1.501243e+02,4.039010e+01,0.0002,***
C(Q('Primary Property Type'))[T.other/specialty hospital],9.340750e+01,2.762650e+01,0.0007,***
Post,-2.542498e+11,1.301036e+12,0.8451,
LowRating,-2.454190e+01,1.705790e+01,0.1502,
Interaction,5.595310e+01,2.892490e+01,0.0531,*


### Multifamily Housing/Office

In [12]:
def make_concurrent_panel(
    panel: pd.DataFrame, id_col: str, year_col: str = "Data Year"
) -> pd.DataFrame:
    """Return a balanced (concurrent) panel where each building appears in all years."""
    panel = panel.copy()
    panel[year_col] = pd.to_numeric(panel[year_col], errors="coerce")
    panel = panel.dropna(subset=[id_col, year_col])
    panel[year_col] = panel[year_col].astype("int64")

    years = sorted(panel[year_col].unique())
    n_years = len(years)

    n_by_building = panel.groupby(id_col)[year_col].nunique()
    keep_ids = n_by_building[n_by_building == n_years].index
    return panel[panel[id_col].isin(keep_ids)].copy()

In [ ]:
mf_df = su.filter_property_type(did_df, "multifamily")
mf_df = su.prep_for_did_levels(mf_df)
did_df = did_df[did_df["Site EUI (kBtu/sq ft)"] < MAX_SITE_EUI]

In [ ]:
id_col = "ID" if "ID" in did_df.columns else "Address"

office_df = su.filter_property_type(did_df, "office")
office_panel = office_df[
    (office_df["Data Year"] >= star_year_I) & (office_df["Data Year"] <= end_year)
].copy()

model_office = su.run_did_regression(
    office_panel,
    "Site EUI (kBtu/sq ft)",
    include_data_year=True,
)

su.summarize_did_results(model_office, focus_terms=["Post", "LowRating", "Interaction"])

/project/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 14, but rank is 1
  warnings.warn('covariance of constraints does not have full '
2026-03-01 22:33:07,545 [INFO] Showing 3 selected coefficients (policy + energy types).


,coef,std_err,p_value,Significance
Post,1.380088e+12,9.277327e+12,0.8817,
LowRating,2.423150e+01,1.567900e+00,0.0000,***
Interaction,-1.401000e+00,1.851000e+00,0.4491,


In [ ]:
mf_df = su.filter_property_type(did_df, "multifamily")
mf_df = su.prep_for_did_levels(mf_df)

mf_panel = mf_df[
    (mf_df["Data Year"] >= star_year_I) & (mf_df["Data Year"] <= end_year)
].copy()

model_mf = su.run_did_regression(
    mf_panel,
    "Site EUI (kBtu/sq ft)",
    include_data_year=True,
)

su.summarize_did_results(model_mf, focus_terms=["Post", "LowRating", "Interaction"])

/project/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 16, but rank is 1
  warnings.warn('covariance of constraints does not have full '
2026-03-01 22:48:23,152 [INFO] Showing 3 selected coefficients (policy + energy types).


,coef,std_err,p_value,Significance
Post,-4.608761e+12,5.905086e+12,0.4351,
LowRating,2.515080e+01,7.600000e-01,0.0000,***
Interaction,-2.177000e+00,7.977000e-01,0.0064,***


#### did plots

In [88]:
df_plot = did_df.copy()
df_plot["TreatedCity"] = (df_plot["City"] == "Chicago").astype(int)

df_plot["Data Year"] = pd.to_numeric(df_plot["Data Year"], errors="coerce")
df_plot["Site EUI (kBtu/sq ft)"] = pd.to_numeric(
    df_plot["Site EUI (kBtu/sq ft)"], errors="coerce"
)

df_plot = df_plot.dropna(subset=["Data Year", "Site EUI (kBtu/sq ft)", "TreatedCity"])

pu.plot_did_trend(
    df_plot,
    year_col="Data Year",
    group_col="TreatedCity",
    outcome_col="Site EUI (kBtu/sq ft)",
    policy_year=2019,
    group_labels={1: "Chicago (treated)", 0: "Other Cities (control)"},
    title="DiD: Site EUI — Chicago vs Other Cities",
)

alt.LayerChart(...)

In [ ]:
pu.plot_did_trend(
    office_panel,
    year_col="Data Year",
    group_col="LowRating",
    outcome_col="Site EUI (kBtu/sq ft)",
    policy_year=2019,
    group_labels={1: "Chicago (treated)", 0: "Other Cities (control)"},
    title="DiD: Site EUI — Office Buildings (Regression Sample)",
)

alt.LayerChart(...)

In [ ]:
pu.plot_did_trend(
    mf_panel,
    year_col="Data Year",
    group_col="LowRating",
    outcome_col="Site EUI (kBtu/sq ft)",
    policy_year=2019,
    group_labels={1: "Chicago (treated)", 0: "Other Cities (control)"},
    title="DiD: Site EUI — Multifamily Housing",
)

alt.LayerChart(...)

#### Sensitivity test - years

In [ ]:
office_df = su.filter_property_type(did_df, "office")
office_panel_18 = office_df[
    (office_df["Data Year"] >= star_year_II) & (office_df["Data Year"] <= end_year)
].copy()

model_office = su.run_did_regression(
    office_panel_18,
    "Site EUI (kBtu/sq ft)",
    include_data_year=True,
)

su.summarize_did_results(model_office, focus_terms=["Post", "LowRating", "Interaction"])

/project/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 13, but rank is 12
  warnings.warn('covariance of constraints does not have full '
2026-03-01 22:47:29,856 [INFO] Showing 3 selected coefficients (policy + energy types).


,coef,std_err,p_value,Significance
Post,-8.1698,0.9316,0.0000,***
LowRating,26.7468,1.9359,0.0000,***
Interaction,-3.4477,2.1638,0.1111,


In [ ]:
mf_df = su.filter_property_type(did_df, "multifamily")
mf_df = su.prep_for_did_levels(mf_df)

mf_panel_18 = mf_df[
    (mf_df["Data Year"] >= star_year_I) & (mf_df["Data Year"] <= end_year)
].copy()

model_mf = su.run_did_regression(
    mf_panel_18,
    "Site EUI (kBtu/sq ft)",
    include_data_year=True,
)

su.summarize_did_results(model_mf, focus_terms=["Post", "LowRating", "Interaction"])

/project/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 15, but rank is 1
  warnings.warn('covariance of constraints does not have full '
2026-03-01 22:49:14,956 [INFO] Showing 3 selected coefficients (policy + energy types).


,coef,std_err,p_value,Significance
Post,1.579960e+12,2.116746e+12,0.4554,
LowRating,2.843060e+01,8.919000e-01,0.0000,***
Interaction,-4.654600e+00,9.186000e-01,0.0000,***


### only compare to Boston

In [ ]:
mf_df = mf_df[mf_df["City"].isin(["Chicago", "Boston"])].copy()

mf_panel_cb = mf_df[
    (mf_df["Data Year"] >= star_year_II) & (mf_df["Data Year"] <= end_year)
].copy()

mf_model_cb = su.run_did_regression(
    mf_panel_cb,
    "Site EUI (kBtu/sq ft)",
    include_data_year=True,
)

su.summarize_did_results(mf_model_cb, focus_terms=["Post", "LowRating", "Interaction"])

/project/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 12, but rank is 1
  warnings.warn('covariance of constraints does not have full '
2026-03-01 22:54:30,857 [INFO] Showing 3 selected coefficients (policy + energy types).


,coef,std_err,p_value,Significance
Post,1.876347e+12,3.241424e+12,0.5627,
LowRating,1.678140e+01,1.389300e+00,0.0000,***
Interaction,2.267500e+00,1.586000e+00,0.1528,


In [117]:
mf_panel_cb["TreatedCity"] = (mf_panel_cb["City"] == "Chicago").astype(int)

pu.plot_did_trend(
    mf_panel_cb,
    year_col="Data Year",
    group_col="TreatedCity",
    outcome_col="Site EUI (kBtu/sq ft)",
    policy_year=2019,
    group_labels={1: "Chicago (treated)", 0: "Boston (control)"},
    title="DiD: Site EUI — Chicago vs Boston (Office, 2017–2023)",
)

alt.LayerChart(...)

### only compare to SF + Seattle

In [ ]:
mf_df = su.filter_property_type(did_df, "multifamily")
mf_df = su.prep_for_did_levels(mf_df)

mf_df = mf_df[mf_df["City"].isin(["Chicago", "San Francisco", "Seattle"])].copy()

mf_panel = mf_df[
    (mf_df["Data Year"] >= star_year_II) & (mf_df["Data Year"] <= end_year)
].copy()

mf_panel = mf_panel.dropna(subset=["Site EUI (kBtu/sq ft)"])

model_mf_west = su.run_did_regression(
    mf_panel,
    "Site EUI (kBtu/sq ft)",
    include_data_year=True,
)

su.summarize_did_results(
    model_mf_west, focus_terms=["Post", "LowRating", "Interaction"]
)

/project/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 15, but rank is 1
  warnings.warn('covariance of constraints does not have full '
2026-03-01 23:31:02,713 [INFO] Showing 3 selected coefficients (policy + energy types).


,coef,std_err,p_value,Significance
Post,9.728344e+12,1.443074e+13,0.5002,
LowRating,4.396580e+01,8.544000e-01,0.0000,***
Interaction,-7.660400e+00,8.540000e-01,0.0000,***


In [127]:
mf_panel["TreatedCity"] = (mf_panel["City"] == "Chicago").astype(int)

pu.plot_did_trend(
    mf_panel,
    year_col="Data Year",
    group_col="TreatedCity",
    outcome_col="Site EUI (kBtu/sq ft)",
    policy_year=2019,
    group_labels={1: "Chicago (treated)", 0: "SF + Seattle (control)"},
    title="DiD: Site EUI — MF (Chicago vs SF + Seattle, 2018–2023)",
)

alt.LayerChart(...)